# Biomedical Methodology Profile Pipeline — v0.6.0

Extracts methodology dimensions from titles and abstracts using an auditable
YAML rulebook, then **optionally** enriches the six intent/context dimensions
with an LLM second opinion that is verified and reconciled against the rules.

**Run the cells in order.** Sections 1-9 are the rule pipeline; 10-13 are the
LLM layer and are entirely optional — the rule pipeline stands alone.

| | |
|---|---|
| 1 | Setup — mount Drive, install, import |
| 2 | Rulebook — load the ontology, optionally patch it |
| 3 | Corpus — load records, map your column names |
| 4 | Smoke test — 500 records first, always |
| 5 | Full run — chunked and resumable |
| 6 | Explain — why did *this* record get these labels? |
| 7 | Audit — lint the rulebook, see what each label matched |
| 8 | Coverage — what fraction did we actually characterise? |
| 9 | Annotation — build the gold-standard worksheet |
| 10 | **LLM setup** — keys, install, connectivity check |
| 11 | **LLM extract** — route low-confidence records, run GPT-4o |
| 12 | **Verify + reconcile** — drop hallucinations, agreement matrix |
| 13 | **Enrich** — merge validated LLM labels into the profile |

Read `ARCHITECTURE.md` and `LLM_LAYER.md` before changing anything.


---
## 1 · Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
pd.set_option('display.max_columns', None)
# display(df)

In [ ]:
!pip -q install pyahocorasick pyarrow pyyaml pandas scikit-learn
# pyahocorasick is optional but gives ~10-20x on candidate selection at
# corpus scale. Everything falls back to str.find without it.


Point `PKG_DIR` at the folder containing `mpp/`, `config/` and `tests/`.
Upload the v0.5 zip to Drive and unzip it there, or clone it.


In [ ]:
from pathlib import Path
import sys, os, time

# PKG_DIR  = Path('/content/drive/MyDrive/methodology_pipeline_v5')   # <-- EDIT
PKG_DIR  = Path('/content/drive/MyDrive/COGNOSX/ticks/code/Biomedical_Methodology_Profile_Colab/methodology_pipeline_v6b')

WORK_DIR = Path('/content/work')      # local scratch — NOT Drive (see below)
OUT_DIR  = PKG_DIR / 'outputs'        # small final artefacts go to Drive

assert PKG_DIR.exists(), f'not found: {PKG_DIR}'
sys.path.insert(0, str(PKG_DIR))
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import mpp
print('mpp', mpp.__version__)


> **Why scratch on local disk, not Drive.** Drive is slow and flaky for many
> small writes, and the chunked runner writes one Parquet part per chunk. Run
> against `/content/work`, then copy the finished artefacts to Drive once.


Run the test suite before trusting anything. 48 regression tests, each
pinning one real defect found in this corpus.


In [ ]:
!cd {PKG_DIR} && python -m pytest tests/ -q 2>&1 | tail -3


---
## 2 · Rulebook


In [ ]:
from mpp import load_ontology

ONTOLOGY = PKG_DIR / 'config' / 'ontology.yml'
onto = load_ontology(ONTOLOGY)

print(f'rule_version {onto.version}')
print(f'{len(onto.specs)} labels across {len(onto.dimensions)} dimensions')
print(f'sources: {list(onto.sources)}')


### Optional — apply the confirmed-fix patch

`config/patch_v0_4_1.yml` fixes defects verified against this corpus:
`\bDimensions\b` matching the English word, `editorial_commentary` firing on
"from a One Health perspective", `guinea pigs` counted as livestock, tick
life stages counted as a human adult population, plus 24 missing
`evidence_category` mappings (animal models currently contribute nothing).

It backs up your ontology first and skips anything naming a label it cannot
find. **Run `--dry-run` first and read the report.**


In [ ]:
!cd {PKG_DIR} && python apply_ontology_patch.py config/ontology.yml config/patch_v0_4_1.yml --dry-run


In [ ]:
# Uncomment to apply, then RE-LOAD the ontology and re-run the tests.
!cd {PKG_DIR} && python apply_ontology_patch.py config/ontology.yml config/patch_v0_4_1.yml
onto = load_ontology(ONTOLOGY); print('now', onto.version)


---
## 3 · Corpus

The pipeline needs `openalex_id` plus at least one source column
(`title`, `abstract`, `keywords_author`, `keywords_openalex`). Missing source
columns are skipped silently, so a corpus without keywords behaves as before.


In [ ]:
# CORPUS = Path('/content/drive/MyDrive/.../screened_llm.parquet')   # <-- EDIT
CORPUS = Path('/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet')  # <-- EDIT

raw_all = pd.read_parquet(CORPUS)
raw = raw_all[raw_all["llm_label"].notna()]
print(f'{len(raw_all):,} records, {len(raw_all.columns)} columns')
print(f'{len(raw):,} records, {len(raw.columns)} columns')
print(list(raw.columns))


Map your column names onto the ones the pipeline expects.

**Keep only what you need.** Carrying all 75 source columns through is what
produced a 586 MB wide table last time. `openalex_id` is the join key — put
the rest back afterwards.


In [ ]:
COLUMN_MAP = {
    'oa_id':       'openalex_id',
    'oa_title':    'title',
    'oa_abstract': 'abstract',
    'author_keywords':   'keywords_author',
    'openalex_keywords': 'keywords_openalex',
}

records = raw.rename(columns=COLUMN_MAP)
keep = [c for c in ['openalex_id','title','abstract',
                    'keywords_author','keywords_openalex'] if c in records.columns]
records = records[keep].copy()
records['openalex_id'] = records['openalex_id'].astype(str)

assert 'openalex_id' in records.columns, 'no openalex_id after mapping'
assert records.openalex_id.is_unique, 'duplicate openalex_id — dedupe first'

print(f'{len(records):,} records | sources present: '
      f'{[c for c in keep if c != "openalex_id"]}')
for c in keep[1:]:
    empty = records[c].isna() | records[c].astype(str).str.strip().eq('')
    print(f'  {c:20} empty in {empty.mean():5.1%}')


> That last number matters. Records with no abstract get a title-only profile,
> which is thin. If the share is large, adding keyword columns is the highest-
> value change available to you.


---
## 4 · Smoke test

**Always run this before the full corpus.** Sixty seconds here beats
discovering a column-mapping mistake twenty-five minutes in.


In [ ]:
from mpp import run_pipeline

sample = records.sample(n=min(500, len(records)), random_state=0)

# Fill NaN values in text columns with empty strings to prevent TypeError
text_columns = ['title', 'abstract', 'keywords_author', 'keywords_openalex']
for col in text_columns:
    if col in sample.columns:
        sample[col] = sample[col].fillna('')

t0 = time.time(); res = run_pipeline(sample, onto); el = time.time() - t0

rate = len(sample) / el
print(f'{rate:,.0f} rec/s  ->  ~{len(records)/rate/60:.0f} min for the full corpus')
print(f'{len(res.mentions):,} mentions -> {len(res.long):,} long rows')

dims = [d for d in onto.dimensions + ['evidence_category'] if d in res.wide.columns]
cov = pd.DataFrame({
    'labelled': [res.wide[d].apply(len).gt(0).mean() for d in dims]},
    index=dims).sort_values('labelled', ascending=False)
cov.style.format('{:.1%}')


In [ ]:
# Sanity checks worth running on every corpus.
print('context distribution (prose sources only):')
prose = res.long[res.long.text_source.isin(['title','abstract'])]
print((prose.mention_context.value_counts(normalize=True)*100).round(1).to_string())
print()
print('how context was decided:')
print((prose.context_evidence.str.split(':').str[0]
       .value_counts(normalize=True)*100).round(1).to_string())
print()
print('top labels:')
print(res.long.groupby(['dimension','label']).size()
      .sort_values(ascending=False).head(15).to_string())


> A single label dominating the counts is almost always a pattern that is too
> permissive, not a real finding. Check it in section 7 before believing it.


---
## 5 · Full run

Chunked and resumable: one Parquet part per chunk, completed parts skipped on
re-run. If Colab disconnects, re-run this cell and it picks up where it stopped.

Drop `'mentions'` from `tables` if you do not need per-hit forensics — it is by
far the largest table and can be regenerated for a subset later.


In [ ]:
from mpp.batch import run_chunked, load_parts, manifest

RUN_DIR = WORK_DIR / f'run_{onto.version}'

# Fill NaN values in text columns with empty strings for the full records DataFrame
text_columns = ['title', 'abstract', 'keywords_author', 'keywords_openalex']
for col in text_columns:
    if col in records.columns:
        records[col] = records[col].fillna('')

meta = run_chunked(records, onto, RUN_DIR,
                   chunk_size=20_000,
                   tables=('mentions','long','wide'))
meta

In [ ]:
long_df = load_parts(RUN_DIR, 'long')
wide_df = load_parts(RUN_DIR, 'wide')
print(f'{len(long_df):,} long rows | {len(wide_df):,} records')

# Slim the wide table before it goes to Drive: id + derived columns only.
dim_cols = [c for c in wide_df.columns
            if c in onto.dimensions + ['evidence_category']
            or c.endswith('__confidence')]
slim = wide_df[['openalex_id'] + dim_cols + ['maximum_extraction_confidence']]
slim.to_parquet(OUT_DIR / 'methodology_wide.parquet', index=False)
long_df.to_parquet(OUT_DIR / 'methodology_long.parquet', index=False)
print('written to', OUT_DIR)
print('wide size:',
      round((OUT_DIR/'methodology_wide.parquet').stat().st_size/1e6, 1), 'MB')


> Rejoin the rest of your metadata on `openalex_id` when you need it. Storing
> it inside the wide table is what made the file unmanageable last time.


---
## 6 · Explain — the cell you will use most

When a result looks wrong, **do not open the YAML.** Ask the pipeline what
happened to that record.

It prints every candidate match, how its context was decided, the score
arithmetic term by term, and for anything rejected the specific reason.


In [ ]:
from mpp.explain import explain, explain_missing, explain_pattern

rid = records.openalex_id.iloc[0]     # <-- any record id
print(explain(records, onto, rid))


The **DROPPED** section is the useful half. "Why does this paper have no
study design?" has five answers needing five different fixes:

| reason | what to do |
|---|---|
| `negated` | usually correct — check the cue is not over-firing |
| context gate | label needs Methods/Results; the abstract is unstructured |
| source gate | evidence came from a field this label may not use |
| review reclassified | correct — designs belong to the *included* studies |
| nothing matched | a genuine rulebook gap; the pattern needs work |

**Only the last is an ontology edit.** Reaching for the YAML on the other four
is how a rulebook fills up with patterns papering over a gating problem.


In [ ]:
# Is a label missing because no pattern fired, or because a gate removed it?
print(explain_missing(records, onto, rid, 'study_design'))


In [ ]:
# Test a rule edit in seconds instead of a 25-minute corpus run.
explain_pattern(onto, 'population', 'adults', [
    'We enrolled 220 adults aged 18-64 in a seroprevalence survey.',
    'In total 2942 ticks (53.9% larvae, 40.1% nymphs, 6.0% adults) were sampled.',
    'Dogs were infested with 30 unfed, adult, female I. holocyclus.',
])


---
## 7 · Audit — the line-by-line review

A 66 KB rulebook cannot be reviewed by reading it. Every defect found in this
project was invisible on the page and obvious in the data: `\bDimensions\b`
looked like a database name; `chi_square` was double-escaped and had matched
**nothing** in 334k records, which looks exactly like a rare label.

Two tables replace the reading pass.


In [ ]:
from mpp.audit import save_audit, lint_ontology, review_worksheet

lint = lint_ontology(onto)
print(lint.groupby(['severity','check']).size().to_string())
lint[lint.severity.isin(['critical','high'])][['check','dimension','label','pattern']]


In [ ]:
ws = review_worksheet(onto, mentions=load_parts(RUN_DIR,'mentions'), long=long_df)
ws.to_csv(OUT_DIR / 'ontology_review_worksheet.csv', index=False)

# THE column to read: what each label actually caught, with counts.
ws[ws.papers > 0][['dimension','label','papers','mean_confidence',
                   'example_matches','status']].head(40)


**How to use the worksheet** (about an hour, and it *is* the line-by-line review):

1. Sort by `papers` descending, read `example_matches` for the top 40 labels.
2. `status == 'DEAD — never matched'` — broken pattern, or a concept absent
   from your corpus. You cannot tell these apart by reading.
3. High `gate_fail_rate` — the *gate*, not the pattern, is setting the count.
4. Sort by `review_priority` (`papers × (1 − mean_confidence)`) — that is your
   annotation queue, already ordered.

Keep each worksheet between versions. A change in a label's `papers` or
`example_matches` is then evidence, not a guess — which is what `rule_version`
is supposed to mean.


---
## 8 · Coverage


In [ ]:
dims = [d for d in onto.dimensions + ['evidence_category'] if d in slim.columns]
cover = pd.DataFrame({
    'labelled_records': [int(slim[d].apply(len).gt(0).sum()) for d in dims],
    'share':            [slim[d].apply(len).gt(0).mean() for d in dims],
}, index=dims).sort_values('share', ascending=False)
cover.style.format({'labelled_records':'{:,}','share':'{:.1%}'})


In [ ]:
# Where is evidence coming from, and how much is each gate costing?
print(long_df.text_source.value_counts().to_string())
print()
for g in ['negated','context_gate_failed','source_gate_failed']:
    if g in long_df.columns:
        print(f'{g:22} {long_df[g].mean():6.1%} of rows')


> Records with **no label in any dimension** are the ones to sample and read.
> That set is what tells you whether the gap is missing patterns, over-
> aggressive gating, or records with genuinely no extractable methodology —
> three problems with three different fixes.


In [ ]:
no_label = slim[dims].apply(lambda row: all(len(v) == 0 for v in row), axis=1)
unlabelled = slim.loc[no_label, 'openalex_id']
print(f'{len(unlabelled):,} records ({no_label.mean():.1%}) have no label in any dimension')

audit_set = (records[records.openalex_id.isin(unlabelled)]
             .sample(n=min(200, len(unlabelled)), random_state=0))
audit_set.to_csv(OUT_DIR / 'unlabelled_audit.csv', index=False)
audit_set.head()


---
## 9 · Annotation

Two strata, because one sample cannot estimate both things:

- **random** — uniform. Estimates prevalence and **recall**.
- **stratified** — up to N rule-positives per label. Estimates **precision**.
  Without it, rare labels get two or three examples in a uniform draw and
  their per-label F1 is noise dressed as a number.

20% of rows are flagged `double_coded`. Run agreement on those **first**: if
two annotators cannot agree what a label means, the definition is the problem,
not the regex, and no pattern tuning will fix it.


In [ ]:
from mpp import make_annotation_template

tpl = make_annotation_template(records, long_df, n_random=300, per_label=15)
tpl.to_csv(OUT_DIR / 'annotation_template.csv', index=False)
print(tpl.stratum.value_counts().to_dict(), '| double-coded:', int(tpl.double_coded.sum()))
tpl.head()


In [ ]:
# After annotation comes back:
from src.evaluation import (agreement_report, evaluate_dimension,
                            calibration_table, silver_standard_comparison)

# gold = pd.read_csv(OUT_DIR / 'annotation_template_completed.csv')
# agreement_report(gold[gold.double_coded], 'study_design_gold_a', 'study_design_gold_b')
# evaluate_dimension(gold, 'study_design_gold', 'study_design')
# calibration_table(long_df, gold_long)


### Validate at scale for free

MEDLINE tags publication types on millions of records. With PMIDs on a third
of the corpus that is tens of thousands of validation rows for `study_design`
and `evidence_synthesis_type` at zero annotation cost — then human-annotate
only the disagreements.

Caveats to state in the paper: indexing lag, the Systematic Review type dates
from 2019, and the tags are themselves imperfect. This is a *silver*, not a
gold, standard.

This is also why MeSH is deliberately **not** a pipeline input: the moment it
becomes one, this validation is circular.


In [ ]:
# pt = pd.read_csv('pubmed_publication_types.csv')   # [pmid, publication_types]
# silver_standard_comparison(slim.merge(raw[['oa_id','pmid']].rename(
#     columns={'oa_id':'openalex_id'}), on='openalex_id'), pt)


---
# LLM enrichment (sections 10-13) — optional

A **second, independent extractor** that adds recall on the six dimensions
where intent and context decide the label (`study_design`, `research_purpose`,
`population`, `evidence_synthesis_type`, `biological_model`, `data_source`).

The rules stay authoritative. Every LLM label must quote a verbatim span from
the text; spans that aren't actually there are dropped as hallucinations. Then
LLM and rule labels are reconciled into an agreement matrix — which is your
validation story, not a byproduct.

**Primary = GPT-4o. DeepSeek = tie-break. Gemini = alternate primary.**

Read `LLM_LAYER.md` for the full rationale and cost model.


## 10 · LLM setup


In [ ]:
!pip -q install openai nest_asyncio ftfy
# google-generativeai only if you'll use Gemini as primary:
!pip -q install google-generativeai


### Load your API keys from Colab Secrets

In the **Secrets** panel (key icon, left sidebar): add your keys and — this is
the step people miss — **toggle 'Notebook access' ON** for each one. Then this
cell copies them into environment variables, which is what the client reads.

Secret names are case- and space-sensitive. Edit the strings below to match the
exact names in your panel.


In [ ]:
import os
from google.colab import userdata

# left = the client's env var (do not change) ; right = YOUR secret's name
SECRET_NAMES = {
    'OPENAI_API_KEY':   'OPENAI_API_KEY',
    'DEEPSEEK_API_KEY': 'DEEPSEEK_API_KEY',
    'GEMINI_API_KEY':   'GEMINI_API_KEY',    # only if using Gemini
}

for env_var, secret_name in SECRET_NAMES.items():
    try:
        os.environ[env_var] = userdata.get(secret_name)
        print(f'  loaded {env_var}')
    except Exception as e:
        print(f'  MISSING {secret_name} -> toggle it ON in Secrets  ({type(e).__name__})')


> Only the **primary** key is required (OpenAI by default). DeepSeek is needed
> only if you run the tie-break; Gemini only if it's your primary. A key stored
> but not toggled on raises a permission error above — flip the toggle and re-run.


In [ ]:
# Connectivity check — a fraction of a cent, catches a bad paste or billing issue
from mpp.llm import LLMClient

PROVIDER = 'openai'          # 'openai' | 'gemini' | 'deepseek'
client = LLMClient(PROVIDER)
print('model:', client.model)

probe = client.extract(
    'Reply with valid JSON only.',
    'Return {"ok": true}',
    {'type':'object','properties':{'ok':{'type':'boolean'}},
     'required':['ok'],'additionalProperties':False})
print('AUTH OK' if probe.ok else f'FAILED: {probe.error}')


## 11 · LLM extract

**Route first.** Do not call the LLM on records the rules already handled — that
is the whole cost lever. Send it only records where a target dimension is empty
or low-confidence. Start with the simplest rule: records with no `study_design`.


In [ ]:
import mpp.llm
print("records_needing_llm" in dir(mpp.llm))   # must be True

In [ ]:
import pandas as pd

def records_needing_llm(rule_long, records, dimensions, min_confidence=0.0):
    df = rule_long
    keep = pd.Series(True, index=df.index)
    for flag in ("negated", "context_gate_failed", "source_gate_failed"):
        if flag in df.columns:
            keep &= ~df[flag].fillna(False).astype(bool)
    if min_confidence and "confidence" in df.columns:
        keep &= df["confidence"].fillna(0) >= min_confidence
    handled = set(df.loc[keep & df["dimension"].isin(list(dimensions)),
                         "openalex_id"].astype(str))
    return records[~records["openalex_id"].astype(str).isin(handled)].copy()

In [ ]:
from mpp.llm import DEFAULT_LLM_DIMENSIONS#, records_needing_llm
from mpp.llm import ExtractionConfig, run_extraction, load_results
print('LLM covers:', DEFAULT_LLM_DIMENSIONS)

# Route: records the rules left with NO usable label in ANY target dimension.
# This is the cost lever — the LLM runs only where the rules produced nothing.
need = records_needing_llm(long_df, records, DEFAULT_LLM_DIMENSIONS)
print(f'{len(need):,} of {len(records):,} records need the LLM ({len(need)/max(len(records),1):.0%})')

# To route more aggressively (also re-do LOW-confidence labels), pass a floor:
need = records_needing_llm(long_df, records, DEFAULT_LLM_DIMENSIONS, min_confidence=0.85)


In [ ]:
from mpp.llm import DEFAULT_LLM_DIMENSIONS#, records_needing_llm
from mpp.llm import ExtractionConfig, run_extraction, load_results
print('LLM covers:', DEFAULT_LLM_DIMENSIONS)

# Route: records the rules left with NO usable label in ANY target dimension.
# This is the cost lever — the LLM runs only where the rules produced nothing.
need = records_needing_llm(long_df, records, DEFAULT_LLM_DIMENSIONS)
print(f'{len(need):,} of {len(records):,} records need the LLM ({len(need)/max(len(records),1):.0%})')

# To route more aggressively (also re-do LOW-confidence labels), pass a floor:
need = records_needing_llm(long_df, records, DEFAULT_LLM_DIMENSIONS, min_confidence=0.85)


> **Cost sanity check before you spend.** Estimate on ~200 records first, read
> the token total, multiply. Get live per-token pricing — abstract length is the
> driver and the numbers move.


In [ ]:
# Optional dry-run cost probe on a small sample
import ftfy
sample = need.sample(n=min(200, len(need)), random_state=0).copy()
for col in ['title','abstract']:
    sample[col] = sample[col].fillna('').map(ftfy.fix_text)   # fix mojibake FIRST

cfg = ExtractionConfig(provider=PROVIDER, concurrency=8)
run_extraction(sample, onto, cfg, out_dir=str(WORK_DIR/'llm_probe'))
_, tok = load_results(str(WORK_DIR/'llm_probe'))
per_rec = (tok['prompt']+tok['completion']) / max(len(sample),1)
print(f'~{per_rec:,.0f} tokens/record -> ~{per_rec*len(need):,.0f} tokens for the full {len(need):,}')
print('multiply by your provider per-token price for a cost estimate')


### Full extraction — resumable

Cached (identical input never pays twice), checkpointed (a crash resumes), and
dead-lettered (permanent failures are written out, never silently dropped —
the failure mode that made the old Gemini screening column unreliable).


In [ ]:
import ftfy, nest_asyncio; nest_asyncio.apply()

need['title']    = need['title'].fillna('').map(ftfy.fix_text)
need['abstract'] = need['abstract'].fillna('').map(ftfy.fix_text)

LLM_DIR = WORK_DIR / f'llm_{PROVIDER}'
cfg = ExtractionConfig(provider=PROVIDER, concurrency=8, max_retries=5)
manifest = run_extraction(need, onto, cfg, out_dir=str(LLM_DIR))
manifest


In [ ]:
responses, tokens = load_results(str(LLM_DIR))
print(f'{len(responses):,} records extracted | {tokens["prompt"]+tokens["completion"]:,} tokens')

# any permanent failures?
dead = LLM_DIR / 'llm_dead_letter.jsonl'
if dead.exists():
    import pandas as pd
    dl = pd.read_json(dead, lines=True)
    print(f'{len(dl)} dead-lettered — inspect before trusting coverage:')
    print(dl.head())


## 12 · Verify + reconcile

Verification drops any label whose evidence isn't literally in the text.
Reconciliation classifies every (record, dimension, label) cell as **both**,
**rules_only**, or **llm_only**.


In [ ]:
from mpp.llm import verify_batch, llm_to_long, reconcile, agreement_summary

sources = {str(r.openalex_id): f'{r.title}\n{r.abstract}'
           for r in need.itertuples()}
verified = verify_batch(responses, sources)

hallucination_rate = 1 - verified.verified.mean()
print(f'{len(verified):,} LLM labels | {hallucination_rate:.1%} dropped as unverifiable')
llm_long = llm_to_long(verified)          # verified labels only


In [ ]:
rec = reconcile(long_df, llm_long,
                dimensions=list(DEFAULT_LLM_DIMENSIONS),
                scope_records=need.openalex_id)
summ = agreement_summary(rec)
summ


**How to read this table (it is the validation section of your paper):**

| column | meaning |
|---|---|
| `both` | rules and LLM agree — most trustworthy |
| `rules_only` | rules fired, LLM saw nothing — candidate rule **false positive** |
| `llm_only` | LLM found it, rules missed — candidate **recall gain** |
| `recall_gain_rate` | share of labels the LLM recovered |

`rules_only` is a free precision audit: where a regex fires and GPT-4o reading
the same text disagrees is exactly where to look for over-firing patterns like
the `\bDimensions\b` bug. Sample both `rules_only` and `llm_only` and check
against your gold set **per dimension** before trusting either.


In [ ]:
# Inspect specific disagreements — e.g. what the LLM added that rules missed
gain = rec[(rec.agreement=='llm_only') & (rec.dimension=='study_design')]
gain = gain.merge(verified[['openalex_id','label','llm_evidence','llm_confidence']],
                  on=['openalex_id','label'], how='left')
gain.sort_values('llm_confidence', ascending=False).head(20)


In [ ]:
# And rule-only cells — candidate false positives to audit
rec[rec.agreement=='rules_only'].groupby('dimension').size()


## 13 · Enrich

Merge LLM labels into the rule profile — but **only** for dimensions whose LLM
precision you validated in section 12, and only above a confidence floor. Rules
stay authoritative; provenance is recorded in `origin` (`rules` / `llm` / `both`)
so this is never an opaque overwrite.


In [ ]:
from mpp.llm import merge_into_profile

# EDIT: only the dimensions you validated. Start conservative.
ACCEPT = {'study_design', 'research_purpose'}
FLOOR  = 0.70          # minimum LLM confidence to accept an llm_only label

enriched = merge_into_profile(long_df, llm_long,
                              accept_dimensions=ACCEPT,
                              llm_confidence_floor=FLOOR)
print('origin breakdown:')
print(enriched.origin.value_counts().to_string())

# coverage before vs after, for the accepted dimensions
for d in ACCEPT:
    before = long_df[long_df.dimension.eq(d)].openalex_id.nunique()
    after  = enriched[enriched.dimension.eq(d)].openalex_id.nunique()
    print(f'  {d:18} {before:,} -> {after:,}  (+{after-before:,})')


In [ ]:
enriched.to_parquet(OUT_DIR / 'methodology_enriched.parquet', index=False)
print('written:', OUT_DIR / 'methodology_enriched.parquet')


### Tie-break (optional, later)

The records/dimensions where rules and GPT-4o disagree — and only those — are
where DeepSeek earns its cost. `cells_for_tiebreak(rec)` returns exactly that
set. Don't build the three-way adjudication until section 12 shows you how
often they actually disagree; if it's rare, a simple rule beats another model.


In [ ]:
from mpp.llm import cells_for_tiebreak
tb = cells_for_tiebreak(rec)
print(f'{len(tb):,} record-dimension pairs disagree ({len(tb)/max(len(need),1):.1%} of extracted records)')
print('-> feed these to a DeepSeek run only if that fraction is worth resolving')


---
## Where to go next

1. Apply `patch_v0_4_1.yml` (section 2) and diff the audit worksheet.
2. Fix everything `critical`/`high` in the lint.
3. Run the LLM layer on a **validation sample with gold** first; measure
   precision per dimension; only then set `ACCEPT` in section 13.
4. Deploy the identical runner on the low-confidence subset of the full corpus.
5. Agreement -> precision/recall with CIs -> calibration -> MEDLINE silver standard.

The rule pipeline is authoritative and reproducible. The LLM is a measured,
verified second opinion — never a silent overwrite. Keep it that way and the
whole thing stays defensible to a reviewer.
